# 00 · Core tests (v4)

Runs the `sepsis_core` test-suite. **Run this before any training notebook; do not train until every test is green.**

| Part | Data | What it proves |
|---|---|---|
| A | Synthetic (generated here, no patient data) | The eight required tests plus end-to-end smoke training and the MLflow fix |
| B | Real MIMIC-IV **FIT/VAL only**, subject-level smoke subsample | The same suite on the real schema; cohort counts and split fingerprint match v3 |
| C | Real VAL, v3 checkpoint (optional) | The v4 core reproduces v3's VAL AUROC 0.8073 from the saved checkpoint |

Governance: every cell prints aggregates only (counts, pass/fail, maximum absolute differences). No row-level frame is displayed. LOC / CONF / EXT are never loaded here; the code refuses to.

In [1]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. Pin a commit hash for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):      # Drive is still needed for the data and for checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: add a Colab secret named GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try: _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
    except RuntimeError as e:
        raise RuntimeError("Could not clone the repo. If it is private, add a Colab secret GITHUB_TOKEN (a token with read access to it).") from e
else:
    _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, "Notebooks", "v4", "sepsis_core")):
    raise FileNotFoundError("The repo has no Notebooks/v4/sepsis_core at this ref. Did you push it?")
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]   # re-running this cell picks up a new commit
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
if importlib.util.find_spec("mlflow") is None:                       # optional: only test 10 needs it
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])

import json, time
import numpy as np, torch
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, synthetic, tests
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})")
print("torch", torch.__version__, "| device", engine.RT.device, "| amp", engine.RT.amp, "| grad scaler", engine.RT.scaler)

Mounted at /content/drive
sepsis_core 4.0.0-phase1 @ commit 77d2dc716171 (main)
torch 2.14.1+cu130 | device cuda | amp torch.float16 | grad scaler True


In [2]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST   = True      # Part B uses a ~3% subject-level subsample of FIT/VAL. The tests do not need more.
RUN_REAL     = True      # False = synthetic only (Part A)
RUN_V3_REPRO = False     # Part C: score the v3 checkpoint on ALL of VAL with the v4 core (a few minutes on a GPU). Expect AUROC 0.8073.
if SMOKE_TEST: C.CFG.apply_smoke()
C.CFG.NUM_WORKERS = 0 if engine.RT.device.type == "cpu" else 2
HAVE_REAL = RUN_REAL and os.path.exists(C.PATHS.DATA_PATH)
print("smoke:", C.CFG.SMOKE_TEST, "| real data found:", HAVE_REAL, "| run name:", C.CFG.RUN_NAME)

smoke: True | real data found: True | run name: v4_smoke


## Part A · Synthetic data

Each line is `PASS`/`FAIL` plus the evidence. For tests 1, 4 and 6 the "must not change" quantities are expected to be exactly `0.0` (they run on CPU in fp32).

Test 6 also **reports** two numbers it does not assert (`REPORTED_layers_ge2_...`): in layers ≥ 2 the change in logits for keys *before* the intervened hour is not zero, because the query has already read that hour. See `Docs/DECISIONS.md`, open item O1.

In [3]:
# ============================ CELL 3 · PART A: the suite on synthetic data ============================
t0 = time.time()
RES_SYN = tests.run_all()            # builds its own synthetic cohort (500 subjects) through the real preprocessing path
print(f"\nPart A took {time.time() - t0:.0f}s")

PASS  1 causal leakage (inputs after t cannot move outputs at <= t)
      {"softmax": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 1.8427971601486206}, "sigmoid": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 1.8549350500106812}, "kernel": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 1.8596491813659668}, "kernel_raw": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 0.5058744549751282}, "ka": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 2.1446423530578613}, "preprocessor": {"max_abs_change_rows_le_t0": {"X": 0.0, "M": 0.0, "D": 0.0, "XL": 0.0, "E": 1.8477439880371094e-06}, "future_rows_changed": 11.571365356445312}}
PASS  2 full coverage (each row predicted once; each row trained once per epoch)
      {"one_window_per_stay/eval": {"rows_expected": 3123, "rows_seen": 3123, "each_exactly_once": true, "valid_labels_covered_per_horizon": [true, true, true], "n_windows": 87, "history_rule_ok": true}, "

/content/sepsis-transformer-analysis/Notebooks/v4/sepsis_core/engine.py:186: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step(); release_attn_cache(model)


[test8] ep00 loss=0.3377 aux=0.538 val_loss=0.1596 | AUROC 0.6692 AUPRC 0.1242 | mass 1.000 | lr 2.00e-05 | 2s
PASS  8 balancing touches FIT only
      {"oversampling_refuses_VAL": true, "oversampling_refuses_eval_dataset": true, "added_items": 174, "added_items_are_positive_FIT_windows": true, "every_original_window_still_present_once": true, "smote_refuses_VAL_rows": true, "smote_synthetic_rows": 460, "smote_keeps_original_rows_first": true, "balanced_training_rows_subset_of_FIT": true, "VAL_scored_once_at_natural_distribution": true, "calibration_refuses_FIT_rows": true}
[smoke softmax] arm=softmax seed=42 params=20,617 items/epoch=310 steps/epoch=20 bs=16 lr=1.00e-03 warmup_steps=3 | focal(a=0.880, g=1.0) aux_w=0.05 | select=auroc_mean soup_k=2
[smoke softmax] ep00 loss=0.3007 aux=0.489 val_loss=0.1160 | AUROC 0.5543 AUPRC 0.0471 | mass 1.000 | lr 8.00e-04 | 1s
[smoke softmax] ep01 loss=0.2521 aux=0.466 val_loss=0.1176 | AUROC 0.6641 AUPRC 0.0796 | mass 1.000 | lr 2.89e-04 | 1s
[sm

## Part B · Real data, FIT/VAL only, smoke subsample

Loads the hourly parquet, applies the v3 label fix, **loads** the frozen split (never rebuilds it) and checks its fingerprint, keeps FIT/VAL, then runs the same suite. The cohort counts are compared with v3 before subsampling.

In [4]:
# ============================ CELL 4 · PART B: load real FIT/VAL (aggregates only) ============================
BUNDLE_REAL, DF_VAL, STATIC_REAL = None, None, None
if HAVE_REAL:
    C.require_mounted()
    df, STATIC_REAL, STATS = data.load_hourly()
    print("cohort:", json.dumps(STATS))
    V3_REF = dict(rows=3_966_669, patients=56_075, stays=74_704)
    for k, ref in V3_REF.items():
        print(f"  {k}: {STATS[k]:,} (v3: {ref:,}) {'OK' if STATS[k] == ref else '<-- DIFFERS FROM v3'}")
    print(f"  positive rate: {STATS['pos_rate']:.4%} (v3: 1.1461%)")
    ONSET = data.stay_onset(df)
    n_sept = int(ONSET.notna().sum()); n_early = int((ONSET < C.CFG.EARLY_ONSET_HOURS).sum())
    print(f"septic stays: {n_sept:,} | early-onset (< {C.CFG.EARLY_ONSET_HOURS} h): {n_early:,} ({n_early / max(1, n_sept):.1%}) (v3: 11,731 / 7,659 / 65.3%)")
    split, sha = data.load_frozen_split()
    print("frozen split sha1:", sha, "(expected", C.CFG.SPLIT_SHA1 + ")")
    df, fold_counts = data.attach_folds(df, split, ("FIT", "VAL"), data.FoldGuard("train"))
    print("subjects per fold in the split:", fold_counts, "| loaded folds:", sorted(df['fold'].unique()))
    print(f"stay length: max {int(df.groupby(C.CFG.ID_COL)[C.CFG.HOUR_COL].max().max())} h (window = {C.CFG.MAX_SEQ_LEN} h, coverage = {C.CFG.COVERAGE})")
    if RUN_V3_REPRO: DF_VAL = df[df["fold"] == "VAL"].copy()
    if C.CFG.SMOKE_TEST: df = data.smoke_subsample(df)
    pre = data.Preprocessor(static_cols=STATIC_REAL).fit(df, (df["fold"] == "FIT").to_numpy(), fold=df["fold"].to_numpy())
    store = pre.transform(df)
    fit_idx = np.flatnonzero(store.stay_fold == "FIT"); val_idx = np.flatnonzero(store.stay_fold == "VAL")
    pr, alpha, pw = data.prevalence_alpha(store, fit_idx)
    print(f"stays: FIT={len(fit_idx):,} VAL={len(val_idx):,} | rows={store.n_rows:,} | hour-gap violations={store.gap_violations:,} | "
          f"FIT prevalence per horizon={[round(p, 4) for p in pr]} | focal alpha={alpha:.4f}")
    BUNDLE_REAL = dict(store=store, pre=pre, fit_idx=fit_idx, val_idx=val_idx, onset=ONSET, alpha=alpha, pw=pw, df=df)
else:
    print("Part B skipped: real data not found at", C.PATHS.DATA_PATH, "(or RUN_REAL=False).")

cohort: {"rows_raw": 7379165, "raw_broadcast_pos_rate": 0.46861101493190627, "rows": 3966669, "patients": 56075, "stays": 74704, "pos_rate": 0.011461001661595662}
  rows: 3,966,669 (v3: 3,966,669) OK
  patients: 56,075 (v3: 56,075) OK
  stays: 74,704 (v3: 74,704) OK
  positive rate: 1.1461% (v3: 1.1461%)
septic stays: 11,731 | early-onset (< 12 h): 7,659 (65.3%) (v3: 11,731 / 7,659 / 65.3%)
frozen split sha1: 65441cff1a47 (expected 65441cff1a47)
subjects per fold in the split: {'CONF': 8227, 'EXT': 2571, 'FIT': 33057, 'LOC': 6340, 'VAL': 5880} | loaded folds: ['FIT', 'VAL']
stay length: max 336 h (MAX_SEQ_LEN = 336)
stays: FIT=1,424 VAL=252 | rows=83,843 | hour-gap violations=0 | FIT prevalence per horizon=[0.0102, 0.0095, 0.009] | focal alpha=0.9904


In [5]:
# ============================ CELL 5 · PART B: the suite on real FIT/VAL ============================
RES_REAL = None
if BUNDLE_REAL is not None:
    t0 = time.time()
    RES_REAL = tests.run_all(BUNDLE_REAL)
    print(f"\nPart B took {time.time() - t0:.0f}s")

PASS  1 causal leakage (inputs after t cannot move outputs at <= t)
      {"softmax": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 2.0677452087402344}, "sigmoid": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 1.9797492027282715}, "kernel": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 2.066166400909424}, "kernel_raw": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 0.4943801164627075}, "ka": {"max_abs_change_hours_le_t": 0.0, "max_abs_change_hours_gt_t": 1.7675554752349854}, "preprocessor": {"max_abs_change_rows_le_t0": {"X": 0.0, "M": 0.0, "D": 0.0, "XL": 0.0, "E": 2.384185791015625e-06}, "future_rows_changed": 11.245244026184082}}
PASS  2 full coverage (each row predicted once; each row trained once per epoch)
      {"one_window_per_stay/eval": {"rows_expected": 11563, "rows_seen": 11563, "each_exactly_once": true, "valid_labels_covered_per_horizon": [true, true, true], "n_windows": 252, "history_rule_ok": true}, 

## Part C · Does the v4 core reproduce v3? (optional)

Loads v3's primary checkpoint and its saved FIT-only preprocessor, scores **all of VAL** with the v4 engine in v3's view (most recent 168 h per stay) and compares with v3's 0.8073. Set `RUN_V3_REPRO = True` in Cell 2 to run it.

In [6]:
# ============================ CELL 6 · PART C: v3 checkpoint through the v4 core ============================
V3_REPRO = None
if RUN_V3_REPRO and DF_VAL is not None:
    ckpt = os.path.join(C.PATHS.V3_FINAL_DIR, "best.pt")
    if not os.path.exists(ckpt):
        print("v3 checkpoint not found at", ckpt)
    else:
        ck = torch.load(ckpt, map_location="cpu", weights_only=False)
        pre3 = data.Preprocessor.from_dict(ck["preproc"])                       # v3's own FIT-only statistics
        store3 = pre3.transform(DF_VAL)
        m3 = models.build_model(ck["hp"], pre3.x_mean); m3.load_state_dict(ck["model"]); m3.to(engine.RT.device).eval()
        loader, _ = dataset.make_loader(store3, np.arange(len(store3.starts)), C.CFG.EVAL_BATCH, False, max_len=168, mode="last")
        rows, lg, _ = engine.predict_loader(m3, loader, as_logits=True)
        met = metrics.compute_metrics(store3, rows, lg)
        V3_REPRO = dict(auroc_mean=met["auroc_mean"], auroc_t1=met["auroc_t1"], auroc_t3=met["auroc_t3"], auroc_t6=met["auroc_t6"], v3_reported=0.8073,
                        abs_diff=abs(met["auroc_mean"] - 0.8073), n_stays=int(len(store3.starts)), n_rows=int(len(rows)))
        print(json.dumps(V3_REPRO, indent=1))
        print("REPRODUCED" if V3_REPRO["abs_diff"] < 1e-3 else "<-- DIFFERS from v3 by more than 0.001: tell Claude before training")
        del m3, store3
else:
    print("Part C skipped (RUN_V3_REPRO=False or no real data).")

Part C skipped (RUN_V3_REPRO=False or no real data).


In [7]:
# ============================ CELL 7 · SUMMARY + REPORT ============================
def _summ(res): return None if res is None else dict(passed=sum(r["passed"] for r in res), total=len(res), failed=[r["name"] for r in res if not r["passed"]])
REPORT = dict(time=time.strftime("%Y-%m-%dT%H:%M:%S"), sepsis_core=sepsis_core.__version__, core_commit=CORE_COMMIT, repo_ref=REPO_REF, torch=torch.__version__, device=str(engine.RT.device),
              amp=str(engine.RT.amp), synthetic=_summ(RES_SYN), real_smoke=_summ(RES_REAL), v3_repro=V3_REPRO,
              details=dict(synthetic=RES_SYN, real_smoke=RES_REAL))
out_dir = os.path.join(C.PATHS.WORK_DIR if HAVE_REAL else ".", "reports"); os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, "00_core_tests.json"); json.dump(REPORT, open(out, "w"), indent=1, default=str)
print("synthetic :", REPORT["synthetic"]); print("real smoke:", REPORT["real_smoke"]); print("v3 repro  :", V3_REPRO); print("report saved:", out)
ALL_GREEN = all(s is None or s["passed"] == s["total"] for s in (REPORT["synthetic"], REPORT["real_smoke"]))
print("\nALL GREEN - safe to move to Phase 2" if ALL_GREEN else "\nNOT GREEN - send the output of this cell (aggregates only) back before training anything")

synthetic : {'passed': 10, 'total': 10, 'failed': []}
real smoke: {'passed': 10, 'total': 10, 'failed': []}
v3 repro  : None
report saved: /content/drive/MyDrive/sepsis_v4/reports/00_core_tests.json

ALL GREEN - safe to move to Phase 2
